# U-Net: chạy từng bước và xem kết quả

Notebook này đi qua toàn bộ quy trình supervised của Hùng, mỗi bước hiện kết quả ngay bên dưới:

1. Xem dữ liệu và cách chia tập
2. Train U-Net (log từng epoch, biểu đồ loss/Dice)
3. Xem model dự đoán trên ảnh **validation**
4. Calibrate ngưỡng từ ảnh tốt của validation
5. Đánh giá trên **test** (bảng metric, ảnh đúng/sai)
6. Thử một ảnh bất kỳ

**Cách dùng:** chọn kernel `.venv` (góc trên bên phải), rồi chạy lần lượt từng ô bằng `Shift+Enter`.
Notebook chỉ gọi các lệnh chính thức trong `scripts/` và hàm trong `src/`; logic không nằm ở đây.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image

from industrial_defect_detection.data.manifests import load_manifest_sample, read_manifest
from industrial_defect_detection.runs import load_config, make_run_id, read_json, run_dirs
from industrial_defect_detection.visualization.overlay import GROUND_TRUTH_RGB, blend, panel

pd.set_option("display.precision", 4)
print("Repo:", ROOT)
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "không có, chạy CPU")

## Thiết lập

- `CATEGORY`: `"tile"` hoặc `"metal_nut"`.
- `SMOKE = True`: chỉ chạy 2 epoch × 2 batch để thử đường ống (vài giây). Đặt `False` để train thật.
- `EXISTING_RUN_ID`: điền run ID có sẵn (tên thư mục trong `artifacts/...`) nếu muốn **xem lại** một lần chạy cũ thay vì train mới.

In [ ]:
CATEGORY = "tile"
SMOKE = False
EXISTING_RUN_ID = None   # ví dụ "20261121-150943-seed42-holdout-crack"

CONFIG_PATH = ROOT / f"configs/supervised/unet_resnet18_{CATEGORY}.yaml"
config = load_config(CONFIG_PATH)
DATASET_ROOT = ROOT / config["data"]["dataset_root"]
MANIFEST = ROOT / config["data"]["manifest"]
rows = read_manifest(MANIFEST, DATASET_ROOT)
manifest = pd.DataFrame(rows)


def run(*args):
    """Chạy một script của repo và in output trực tiếp từng dòng."""
    command = [sys.executable, *map(str, args)]
    print(">", " ".join(Path(a).name if i == 0 else a for i, a in enumerate(map(str, command))))
    proc = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, encoding="utf-8", errors="replace")
    for line in proc.stdout:
        if "warning" not in line.lower():
            print(line, end="", flush=True)
    if proc.wait():
        raise RuntimeError("Lệnh bị lỗi, xem output ở trên")


def show(images, titles=None, size=4):
    fig, axes = plt.subplots(1, len(images), figsize=(size * len(images), size))
    for ax, image, title in zip(np.atleast_1d(axes), images, titles or [""] * len(images)):
        ax.imshow(image)
        ax.set_title(title, fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()


print("Config:", CONFIG_PATH.relative_to(ROOT))
print("Manifest:", MANIFEST.relative_to(ROOT), "| giữ riêng lỗi:", config["experiment"]["fold"])

## 1. Dữ liệu và cách chia tập

Bảng dưới đếm số ảnh theo loại (hàng) và tập (cột). Loại lỗi **giữ riêng** chỉ nằm ở cột `test`:
U-Net không bao giờ thấy nó lúc train hay chọn checkpoint.

In [ ]:
table = pd.crosstab(manifest["defect_type"], manifest["split"], margins=True, margins_name="tổng")
held_out = manifest["held_out_defect"].iloc[0]
print("Loại lỗi giữ riêng:", held_out)
table

In [ ]:
# Một ảnh mỗi loại: ảnh gốc và vùng lỗi thật (mask) tô xanh lá.
examples = manifest.groupby("defect_type").head(1)
images, titles = [], []
for row in examples.to_dict("records"):
    image, mask = load_manifest_sample(DATASET_ROOT, row)
    images.append(blend(image, mask, GROUND_TRUTH_RGB))
    titles.append(f"{row['defect_type']} ({row['split']})")
show(images, titles, size=3)

## 2. Train U-Net

Mỗi dòng là một epoch:
- `train_loss` / `val_loss`: BCE + Dice, càng thấp càng tốt.
- `val_dice_defect`: mask dự đoán trùng mask thật bao nhiêu trên ảnh lỗi của validation (1 = trùng hoàn toàn).
- `good_any_pixel`: tỷ lệ ảnh **tốt** của validation bị tô ít nhất một pixel lỗi (báo nhầm sớm).
- Dấu `*`: checkpoint tốt nhất theo `val_dice_defect` được lưu lại.

Với RTX 4050, 20 epoch tile mất khoảng 1 phút.

In [ ]:
seed, fold = config["experiment"]["seed"], config["experiment"]["fold"]
if EXISTING_RUN_ID:
    RUN_ID = EXISTING_RUN_ID
    print("Dùng lại run có sẵn:", RUN_ID)
else:
    RUN_ID = make_run_id(seed, f"smoke-{fold}" if SMOKE else fold)
    run("scripts/train_supervised.py", "--config", CONFIG_PATH, "--run-id", RUN_ID,
        *(["--smoke-test"] if SMOKE else []))
ARTIFACT_DIR, OUTPUT_DIR = run_dirs(config, RUN_ID)
print("Artifact:", ARTIFACT_DIR.relative_to(ROOT))

In [ ]:
history = pd.read_csv(OUTPUT_DIR / "history.csv")
fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
history.plot(x="epoch", y=["train_loss", "val_loss"], ax=axes[0], marker="o", title="Loss")
history.plot(x="epoch", y="val_dice_defect", ax=axes[1], marker="o", title="Val Dice (ảnh lỗi)", legend=False)
history.plot(x="epoch", y="val_good_any_pixel_rate", ax=axes[2], marker="o", title="Tỷ lệ ảnh tốt bị tô (val)", legend=False)
best = history.loc[history["val_dice_defect"].idxmax()]
axes[1].axvline(best["epoch"], color="red", linestyle="--")
plt.tight_layout()
plt.show()
print(f"Checkpoint tốt nhất: epoch {int(best['epoch'])}, val Dice = {best['val_dice_defect']:.4f}")

## 3. Model dự đoán trên ảnh validation

Chưa có ngưỡng chính thức nên mask ở đây dùng tạm xác suất > 0.5. Mỗi hàng:
ảnh gốc | vùng lỗi thật | heatmap xác suất | vùng model tô.

In [ ]:
from industrial_defect_detection.inference.loader import load_predictor

predictor = load_predictor(ARTIFACT_DIR)
val = manifest[manifest["split"] == "val"]
picked = pd.concat([val[val["defect_type"] == "good"].head(2),
                    val[val["defect_type"] != "good"].groupby("defect_type").head(1)])
for row in picked.to_dict("records"):
    image, mask = load_manifest_sample(DATASET_ROOT, row)
    raw = predictor.predict(image)
    title = f"{row['defect_type']}  max prob = {raw.image_score:.3f}"
    show([panel(image, raw.score_map, raw.score_map > 0.5, 0.0, 1.0, mask, title)], size=14)

## 4. Calibrate ngưỡng

Ngưỡng chỉ được chọn từ **ảnh tốt của validation**, theo hợp đồng chung cho cả ba mô hình:
- ngưỡng ảnh = quantile 0.95 của điểm ảnh → khoảng 5% ảnh tốt của val vượt ngưỡng;
- ngưỡng pixel = quantile 0.995 của mọi pixel.

Biểu đồ so sánh điểm của ảnh tốt và ảnh lỗi **trên validation**. Hai nhóm càng tách xa nhau càng dễ phân loại.

In [ ]:
if read_json(ARTIFACT_DIR / "metadata.json").get("status") == "calibrated":
    print("Artifact đã calibrate từ trước, giữ nguyên ngưỡng cũ.")
else:
    run("scripts/calibrate.py", "--artifact-dir", ARTIFACT_DIR, "--dataset-root", DATASET_ROOT,
        "--manifest", MANIFEST)
calibration = read_json(ARTIFACT_DIR / "metadata.json")["calibration"]
print(f"image_threshold = {calibration['image_threshold']:.4f} | pixel_threshold = {calibration['pixel_threshold']:.4f}")

In [ ]:
scores = {"good": [], "lỗi": []}
for row in val.to_dict("records"):
    image, _ = load_manifest_sample(DATASET_ROOT, row)
    scores["lỗi" if row["is_anomalous"] else "good"].append(predictor.predict(image).image_score)
plt.figure(figsize=(8, 3.5))
bins = np.linspace(0, 1, 41)
plt.hist(scores["good"], bins=bins, alpha=0.6, label=f"ảnh tốt (n={len(scores['good'])})")
plt.hist(scores["lỗi"], bins=bins, alpha=0.6, label=f"ảnh lỗi đã biết (n={len(scores['lỗi'])})")
plt.axvline(calibration["image_threshold"], color="red", linestyle="--", label="ngưỡng ảnh")
plt.xlabel("image score (max xác suất)")
plt.ylabel("số ảnh")
plt.title("Validation")
plt.legend()
plt.show()

## 5. Đánh giá trên test

> ⚠️ **Chỉ chạy khi đã khóa config, checkpoint và ngưỡng.** Không quay lại sửa model dựa trên kết quả
> test; nếu có, phải ghi là thăm dò (xem deadline plan). Ô này dùng đúng lệnh `scripts/evaluate.py`
> mà PatchCore cũng sẽ dùng.

In [ ]:
EVAL_DIR = OUTPUT_DIR / f"eval-{MANIFEST.stem}"
if EVAL_DIR.exists():
    print("Đã có kết quả đánh giá, đọc lại:", EVAL_DIR.relative_to(ROOT))
else:
    run("scripts/evaluate.py", "--artifact-dir", ARTIFACT_DIR, "--dataset-root", DATASET_ROOT,
        "--manifest", MANIFEST)
metrics = read_json(EVAL_DIR / "metrics.json")
predictions = pd.read_csv(EVAL_DIR / "predictions.csv")

In [ ]:
image_keys = ["auroc", "average_precision", "precision", "recall", "f1", "false_positive_rate"]
pixel_keys = ["auroc", "average_precision", "dice_defect_images", "iou_defect_images", "dice_good_images"]
print("Số ảnh test:", metrics["counts"]["by_group"])
display(pd.DataFrame({"cấp ảnh": {k: metrics["image"][k] for k in image_keys}}))
display(pd.DataFrame({"cấp pixel": {k: metrics["pixel"][k] for k in pixel_keys}}))
display(pd.DataFrame(metrics["groups"]).drop("undefined").T)

Theo từng loại lỗi: `recall` = tỷ lệ ảnh lỗi bị bắt được. Hàng có `group = held_out` là loại lỗi
U-Net **chưa từng thấy**. Đây là con số chính để so với PatchCore.

In [ ]:
pd.DataFrame(metrics["per_defect_type"]).T

In [ ]:
# Ma trận nhầm lẫn và danh sách ảnh model đoán sai.
print(pd.crosstab(predictions["group"], predictions["is_defective"].map({0: "PASS", 1: "DEFECT"})))
wrong = predictions[predictions["correct"] == 0]
wrong[["sample_id", "group", "image_score", "image_threshold", "is_defective", "dice"]]

In [ ]:
# Xem ảnh: các ảnh đoán sai trước, sau đó mỗi nhóm thêm một ví dụ đúng.
def show_eval(sample_ids):
    for sample_id in sample_ids:
        name = sample_id.replace("/", "__").rsplit(".", 1)[0]
        display(Image.open(EVAL_DIR / "visualizations" / f"{name}.png"))

right = predictions[predictions["correct"] == 1].groupby("group").head(1)
show_eval(list(wrong["sample_id"]) + list(right["sample_id"]))

## 6. Thử một ảnh bất kỳ

Đổi `IMAGE_PATH` sang ảnh bạn muốn (ảnh mới không cần có mask). Kết quả dùng ngưỡng đã lưu trong artifact,
giống cách app demo sẽ làm.

In [ ]:
from industrial_defect_detection.inference.loader import calibrated_predict

predictor = load_predictor(ARTIFACT_DIR)  # nạp lại để đọc ngưỡng vừa calibrate
IMAGE_PATH = DATASET_ROOT / CATEGORY / "test" / held_out / "000.png"
image = np.asarray(Image.open(IMAGE_PATH).convert("RGB"))
prediction = calibrated_predict(predictor, image)
verdict = "DEFECT" if prediction.is_defective else "PASS"
title = f"{verdict}  score={prediction.image_score:.4f}  ngưỡng={prediction.image_threshold:.4f}"
show([panel(image, prediction.score_map, prediction.pred_mask, 0.0, 1.0, title=title)], size=12)